In [ ]:
# import required libraries
import kagglehub
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
# load data set
donwloadPath = kagglehub.dataset_download('lespin/house-prices-dataset')
dataPath = Path(donwloadPath)

print(f'Contents of {dataPath}:')
for item in dataPath.iterdir():
    print(f' - {item.name} ({'Folder' if item.is_dir() else 'File'})')




In [ ]:
# Inspect data set
dfHousing = pd.read_csv(f'{donwloadPath}/train.csv')
originalColumns = dfHousing.columns

# print first 5 lines
print('-- Housing : First few Rows --')
display(dfHousing.head())

# check data set shape
print(f'-- Housing Shape: {dfHousing.shape} --')

# check available data types
print('\n-- Housing Data Type Count --')
print(dfHousing.dtypes.value_counts())

# check data counts (checking missing values)
print('\n-- Housing Info --')
dfHousing.info()

# get statistical summary
print('\n-- Housing Targer Summary (Sales Price) --')
display(dfHousing['SalePrice'].describe())

# visualize distribution
sns.set_theme(style='whitegrid')
plt.figure(figsize=(8,5))
sns.histplot(dfHousing['SalePrice'], kde=True, color='green', bins=30)
plt.title('Distribution of House Sale Prices')
plt.xlabel('Sale Price/($)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Handle missing values

# Check missing values
print('\n-- Missing values before hangling --')
missingData = dfHousing.isnull().sum()
print(missingData[missingData > 0])

# replace with median value
if 'LotFrontage' in dfHousing.columns:
    lotFrontageMedian = dfHousing['LotFrontage'].median()
    dfHousing['LotFrontage'] = dfHousing['LotFrontage'].fillna(lotFrontageMedian)
    print(f"\nFilled missing 'LotFrontage' values with meadian: {lotFrontageMedian}")

# replace with frequent value
if 'Electrical' in dfHousing.columns:
    electricalMode = dfHousing['Electrical'].mode()[0]
    dfHousing['Electrical'] = dfHousing['Electrical'].fillna(electricalMode)
    print(f"Filled missing 'Electrical' values with mode: {electricalMode}")

# replace with 0
if 'MasVnrArea' in dfHousing.columns:
    dfHousing['MasVnrArea'] = dfHousing['MasVnrArea'].fillna(0)
    print("Filled missing 'MasVnrArea' values with value: 0")


# replace all other missing string values with 'None'
attributes = ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']
for attr in attributes:
    if attr in dfHousing.columns:
        dfHousing[attr] = dfHousing[attr].fillna('None')
        print(f"Filled missing '{attr}' values with value: 'None'")       

# add hasGarage attribute
dfHousing['HasGarage'] = (dfHousing['GarageType'] != 'None').astype(int)

if 'GarageYrBlt' in dfHousing.columns:
    dfHousing['GarageYrBlt'] = dfHousing['GarageYrBlt'].fillna(0)
    print("Filled missing 'GarageYrBlt' values with value: 0")

# Validate missing values
print('\n-- Missing values after hangling --')
missingData = dfHousing.isnull().sum()
print(missingData[missingData > 0])

print(dfHousing.shape)



In [ ]:
# Encode categorical variables
categoricalColmns = dfHousing.select_dtypes(include=['str']).columns
for colmn in categoricalColmns:
    uniqueValues = dfHousing[colmn].dropna().unique()
    allValues = ", ".join(map(str, uniqueValues))
    print(f"{colmn} : {allValues}")

In [ ]:
binaryCategory = ['CentralAir']
ordinalCategory = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence']
nominalCategory = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'Electrical', 'Functional', 'GarageType', 'PavedDrive', 'MiscFeature', 'SaleType', 'SaleCondition']

orderMap = {'Ex':5, 'Gd':4, 'GdPrv':4, 'TA':3, 'Av':3, 'Fin':3, 'MnPrv':3, 'Fa':2, 'Mn':2, 'RFn':2, 'GdWo':2, 'Po':1, 'No':1, 'Unf':1, 'MnWw':1, 'None':0 }
addColumns = {'HasBsmt':'BsmtQual', 'HasFirePlace':'FireplaceQu', 'HasPool':'PoolQC', 'HasFence':'Fence'}

for key,value in addColumns.items():
    if key not in dfHousing.columns:
        dfHousing[key] = (dfHousing[value] != 'None').astype(int)

# binary --> binary encoding
for key in binaryCategory:
    if key in dfHousing.columns:
        dfHousing[key] = dfHousing[key].map({'Y':1, 'N':0})

# ordinal --> ordinal encoding
for key in ordinalCategory:
    if key in dfHousing.columns:
        dfHousing[key] = dfHousing[key].map(orderMap)

# nominal --> one-hot encoding
dfHousing = pd.get_dummies(dfHousing, columns=nominalCategory, drop_first=True)



In [ ]:
missingData = dfHousing.isnull().sum()
print(missingData[missingData > 0])

print(dfHousing.shape)
display(dfHousing.head())
dfHousing.info()
print(dfHousing[['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence']].head())

In [ ]:
# numerical features 

numericalColms = []
for numCol in originalColumns:
    if numCol not in categoricalColmns:
        numericalColms.append(numCol)

print(numericalColms)

In [ ]:
# scale numerical features (standarization)
# x_new = x - (mean)/(standard deviation)
numericalColms = ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SalePrice']
scaler = StandardScaler()

print(dfHousing[numericalColms])
for numCol in numericalColms:
    if numCol in dfHousing.columns:
        dfHousing[numCol] = scaler.fit_transform(dfHousing[[numCol]])
print(dfHousing[numericalColms])

print(dfHousing[ordinalCategory])
for numCol in ordinalCategory:
    if numCol in dfHousing.columns:
        dfHousing[numCol] = scaler.fit_transform(dfHousing[[numCol]])

print(dfHousing[ordinalCategory])
